# Finite-Dimensional Relational Clock Model (v1)

This notebook isolates a mathematically explicit finite-dimensional model. It distinguishes:

1. **Exact relational conditioning** from a joint clock–system state.
2. **Classical uncertainty in the clock-phase readout**, modeled as a circular stochastic kernel.
3. **Physical decoherence**, which is *not* assumed or derived here.

The model uses a finite phase basis and a qubit whose energy gap matches an integer clock-frequency multiple. This is a deliberately narrow test case, not a proof of the broader Hilbert-Space Quadrature Conjecture.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
print("NumPy initialized.")


## 1. Define the joint state and clock-phase measurement

Let \(	heta_k=2\pi k/N\). The clock phase states are

\[|	heta_kangle=N^{-1/2}\sum_{n=0}^{N-1}e^{in	heta_k}|nangle.\]

For the simple stationary, energy-matched example, take

\[|\Psiangle=a|0angle_C|0angle_S+b|mangle_C|1angle_S,\qquad |a|^2+|b|^2=1.\]

Here the system gap must equal \(m\hbar\omega\) if the clock levels have energies \(n\hbar\omega\). The exact constraint is therefore restrictive: an arbitrary qubit energy gap does not automatically satisfy it.


In [ ]:
def relational_clock_model(N=128, m=1, a=1/np.sqrt(2), b=1/np.sqrt(2)):
    """Return exact phase-conditioned states from a two-term joint state.

    Convention: |theta_k> = (1/sqrt(N)) sum_n exp(+i*n*theta_k)|n>.
    Then <theta_k|Psi> has the relative phase exp(-i*m*theta_k).
    """
    if N < 2:
        raise ValueError("N must be at least 2")
    if not (0 <= m < N):
        raise ValueError("m must be an integer clock index in [0, N-1]")
    norm = np.sqrt(abs(a)**2 + abs(b)**2)
    if norm == 0:
        raise ValueError("At least one amplitude must be nonzero")
    a, b = a / norm, b / norm
    theta = 2 * np.pi * np.arange(N) / N

    # The phase measurement amplitude is 1/sqrt(N) times this vector.
    conditional_kets = np.column_stack([
        np.full(N, a, dtype=complex),
        b * np.exp(-1j * m * theta)
    ])
    # Each row is normalized conditional |psi(theta_k)>; phases are generated
    # from the joint state, not supplied as a separate ideal trajectory.
    rho = np.einsum('ki,kj->kij', conditional_kets, conditional_kets.conj())
    probabilities = np.full(N, 1.0 / N)
    return theta, conditional_kets, rho, probabilities

theta, kets, rho, p_clock = relational_clock_model()
print("First conditional ket:", kets[0])
print("First conditional density matrix:\n", rho[0])
print("Sum of clock outcome probabilities:", p_clock.sum())


## 2. Add finite phase-readout resolution

Model the *readout uncertainty* using a normalized circular Gaussian kernel \(K_{k\ell}\). Given a reported phase \(	heta_k\), the observed conditional state is a mixture of exact conditional states weighted by the probability that the true phase was \(	heta_\ell\). Since the ideal phase outcomes have equal probabilities in this two-term example,

\[ho_{m obs}(	heta_k)=\sum_\ell K_{k\ell}ho(	heta_\ell).\]

This is a classical measurement-smearing model. It is not, by itself, evidence for intrinsic decoherence or a Lindblad law. The width \(\sigma_	heta\) is an explicit parameter, not yet derived from a microscopic clock model.


In [ ]:
def circular_gaussian_kernel(theta, sigma_theta):
    """Row-stochastic circular Gaussian: row k weights true phases l for readout k."""
    if sigma_theta < 0:
        raise ValueError("sigma_theta must be nonnegative")
    N = len(theta)
    if sigma_theta == 0:
        return np.eye(N)
    # Shortest signed angular difference in [-pi, pi).
    delta = (theta[:, None] - theta[None, :] + np.pi) % (2*np.pi) - np.pi
    weights = np.exp(-0.5 * (delta / sigma_theta)**2)
    weights /= weights.sum(axis=1, keepdims=True)
    return weights

def smear_conditional_states(rho, theta, sigma_theta):
    K = circular_gaussian_kernel(theta, sigma_theta)
    rho_obs = np.einsum('kl,lij->kij', K, rho)
    return K, rho_obs

sigma_theta = 0.20
K, rho_obs = smear_conditional_states(rho, theta, sigma_theta)
print("Maximum kernel row-sum error:", np.max(np.abs(K.sum(axis=1) - 1)))
print("Maximum observed trace error:", np.max(np.abs(np.trace(rho_obs, axis1=1, axis2=2) - 1)))


## 3. Numerical checks

We test normalization, Hermiticity, positivity, phase periodicity, and the zero-resolution limit. These are checks of this finite model and implementation only; they do not establish the manuscript's winding quotient or the conjecture as a whole.


In [ ]:
def run_assertions():
    N, m = 64, 3
    theta, kets, rho, p = relational_clock_model(N=N, m=m, a=0.6, b=0.8j)
    assert np.allclose(p.sum(), 1.0)
    assert np.allclose(np.linalg.norm(kets, axis=1), 1.0)
    assert np.allclose(np.trace(rho, axis1=1, axis2=2), 1.0)
    assert np.allclose(rho, rho.conj().transpose(0, 2, 1))
    eigs = np.linalg.eigvalsh(rho)
    assert eigs.min() > -1e-12

    K0, rho0 = smear_conditional_states(rho, theta, 0.0)
    assert np.allclose(K0, np.eye(N))
    assert np.allclose(rho0, rho)

    K, rho_sm = smear_conditional_states(rho, theta, 0.25)
    assert np.allclose(K.sum(axis=1), 1.0)
    assert np.allclose(rho_sm, rho_sm.conj().transpose(0, 2, 1))
    assert np.allclose(np.trace(rho_sm, axis1=1, axis2=2), 1.0)
    assert np.linalg.eigvalsh(rho_sm).min() > -1e-12

    # Integer m gives a 2pi-periodic phase dependence.
    psi0 = np.array([0.6, 0.8j], dtype=complex)
    psi2pi = np.array([0.6, 0.8j*np.exp(-1j*m*2*np.pi)], dtype=complex)
    assert np.allclose(np.outer(psi0, psi0.conj()), np.outer(psi2pi, psi2pi.conj()))
    return {
        "phase_outcomes": N,
        "energy_gap_index_m": m,
        "min_exact_density_eigenvalue": float(eigs.min()),
        "min_smeared_density_eigenvalue": float(np.linalg.eigvalsh(rho_sm).min()),
        "max_trace_error": float(np.max(np.abs(np.trace(rho_sm, axis1=1, axis2=2)-1))),
        "checks": "passed"
    }

run_assertions()


## 4. Visualize exact and smeared coherence

For the density matrix convention \(ho_{01}=\langle 0|ho|1angle\), the exact off-diagonal element oscillates with the clock phase. Readout smearing reduces its contrast. The contrast loss here follows from the selected classical kernel and should not be labeled physical decoherence without an additional derivation.


In [ ]:
N, m = 128, 2
theta, kets, rho, p = relational_clock_model(N=N, m=m)
sigma_theta = 0.20
K, rho_obs = smear_conditional_states(rho, theta, sigma_theta)

fig, ax = plt.subplots(figsize=(10, 4.8))
ax.plot(theta, rho[:, 0, 1].real, label="Exact conditional Re(rho_01)")
ax.plot(theta, rho_obs[:, 0, 1].real, label="Readout-smeared Re(rho_01)")
ax.set_xlabel("Clock phase theta (radians)")
ax.set_ylabel("Off-diagonal density-matrix element")
ax.set_title(f"Relational qubit phase and readout smearing (N={N}, m={m}, sigma={sigma_theta})")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. What this model establishes—and what it does not

**Established within this toy model:**
- The conditional state is obtained by applying the clock phase measurement to an explicitly defined joint state.
- The conditional density matrices are normalized, positive, and periodic for integer \(m\).
- A normalized circular readout kernel produces normalized mixed conditional states.
- Zero readout uncertainty recovers the exact conditional state.

**Not established:**
- That every system Hamiltonian admits this exact finite clock constraint.
- That the width \(\sigma_	heta\) must equal \(1/(2\Delta n)\); that relation requires a defined clock state and measurement model.
- That the smeared evolution obeys an exact Lindblad equation.
- That the construction proves the manuscript's winding quotient, universality claim, or any quantum-gravity/cosmology extension.

**Next step:** generalize from the two-term energy-matched state to a finite-dimensional system with several energy levels, and state precisely which constraint and clock Hamiltonian are being used. Only after that should we compare the derived predictions with the original notebook.
